In [ ]:
%pip install scikit-optimize

In [16]:
import pandas as pd
import time

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, chi2

from skopt import BayesSearchCV
from skopt.space import Real

In [17]:
real = pd.read_csv("True.csv")
fake = pd.read_csv("Fake.csv")


In [18]:
real["label"] = "Real"
fake["label"] = "Fake"

data = pd.concat([real, fake], ignore_index=True)


In [19]:
X = data["text"]
y = data["label"]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [21]:
tfidf = TfidfVectorizer(max_features=3000)
X_train_tfidf = tfidf.fit_transform(X_train)

In [22]:
# grid search
param_grid = {"C": [0.1, 1, 10]}

start_time = time.time()

grid_search = GridSearchCV(SVC(kernel="linear"), param_grid, cv=2, scoring="accuracy", n_jobs=-1)

grid_search.fit(X_train_tfidf, y_train)
grid_time = time.time() - start_time

In [ ]:
# apply grid search
param_dist = {
    "C": [0.1, 1, 10, 100]
}

start_time = time.time()

random_search = RandomizedSearchCV(SVC(kernel="linear"), param_dist, n_iter=4, cv=2, scoring="accuracy", random_state=42, n_jobs=-1)

random_search.fit(X_train_tfidf, y_train)
random_time = time.time() - start_time

In [ ]:
# apply random search
param_dist = {
    "C": [0.1, 1, 10, 100]
}

start_time = time.time()

random_search = RandomizedSearchCV(SVC(kernel="linear"), param_dist, n_iter=4, cv=2, scoring="accuracy", random_state=42, n_jobs=-1)

random_search.fit(X_train_tfidf, y_train)
random_time = time.time() - start_time

In [ ]:
# apply bayesian optimization
search_space = {
    "C": Real(0.1, 10, prior="log-uniform")
}

start_time = time.time()

bayes_search = BayesSearchCV(SVC(kernel="linear"), search_space, n_iter=4, cv=2, scoring="accuracy", random_state=42, n_jobs=-1)

bayes_search.fit(X_train_tfidf, y_train)
bayes_time = time.time() - start_time

In [ ]:
# compare all three methods
comparison = pd.DataFrame({
    "Method": ["Grid Search", "Random Search", "Bayesian Optimization"],
    "Best Accuracy": [
        grid_search.best_score_,
        random_search.best_score_,
        bayes_search.best_score_
    ],
    "Execution Time": [
        grid_time,
        random_time,
        bayes_time
    ]
})

comparison

In [ ]:
print("Grid Search:", grid_search.best_params_)
print("Random Search:", random_search.best_params_)
print("Bayesian Optimization:", bayes_search.best_params_)